# Chapter 4 — Correlation and Association
**MADT6004 · Brew Lab BKK case**

Correlation measures the strength of a relationship.

| Variables | Method |
|---|---|
| Two continuous (linear) | Pearson |
| Two continuous (monotonic, robust to outliers) | Spearman |
| Two categorical | Chi-square test of independence |

Each method returns a statistic and a p-value. Always **plot first** — a number without a picture can mislead.


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup

In [ ]:
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
from scipy import stats

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


## 2. Build a branch-day panel with attributes
We'll need numeric branch attributes and daily metrics on the same table.

In [ ]:
panel = pd.read_sql("""
SELECT date(t.datetime) AS d, t.branch_id,
       SUM(t.total) AS revenue,
       COUNT(*)     AS orders,
       AVG(t.total) AS avg_ticket
FROM transactions t
GROUP BY t.branch_id, date(t.datetime)
""", conn)
br = pd.read_sql("SELECT branch_id, name, seats, size_sqm, base_traffic FROM branches", conn)
panel = panel.merge(br, on="branch_id")
print(panel.head())


## 3. Pearson — base traffic vs daily revenue
Branches with more foot traffic should earn more revenue. Test it.

In [ ]:
sub = panel[["base_traffic", "revenue"]].dropna()
r, p = stats.pearsonr(sub["base_traffic"], sub["revenue"])
print(f"Pearson r = {r:+.3f}, p = {p:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sub["base_traffic"], sub["revenue"], alpha=0.3, color="#0891B2")
ax.set_xlabel("Base traffic"); ax.set_ylabel("Daily revenue")
ax.set_title(f"Pearson r = {r:+.3f}")
plt.tight_layout(); plt.show()


## 4. Spearman — seats vs orders
When the relationship may not be linear, Spearman (rank correlation) is safer.

In [ ]:
sub = panel[["seats", "orders"]].dropna()
rho, p = stats.spearmanr(sub["seats"], sub["orders"])
print(f"Spearman ρ = {rho:+.3f}, p = {p:.4f}")


## 5. Chi-square — district vs channel preference
Do customers in different districts prefer different ordering channels?

In [ ]:
tx = pd.read_sql("""
SELECT b.district, t.channel
FROM transactions t JOIN branches b ON t.branch_id = b.branch_id
""", conn)
ct = pd.crosstab(tx["district"], tx["channel"])
print(ct)
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"\nχ² = {chi2:.2f}, dof = {dof}, p = {p:.4f}")


## Discussion prompts
1. A high correlation does not mean causation. Give one alternative explanation for the base-traffic ↔ revenue relationship that isn't "more foot traffic causes higher revenue."
2. When would you prefer Spearman over Pearson?
3. The chi-square just tells you *whether* district and channel are related. How would you describe *how* they differ?
